# Nemotron Trace Generator v1 — boxed-format patch

Purpose:
- read `train.csv`
- classify each puzzle into a category
- generate deterministic boxed-answer traces for selected categories
- export an SFT-ready corpus for later training

Scope for v1:
- numeral
- gravity
- unit_conversion

This notebook does **not** train the model and does **not** package a submission.

Patch notes:
- uses a safe `boxed(answer)` helper so Python never converts `\b` into a backspace
- validates literal `\boxed{...}` formatting before writing artifacts
- validates boxed answer exactly matches `train.answer`
- auto-discovers `train.csv` under `/kaggle/input` and prints available CSV files for path debugging


In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
import polars as pl

KAGGLE_INPUT_ROOT = Path('/kaggle/input')

PREFERRED_TRAIN_CANDIDATES = [
    '/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv',
    '/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv',
    '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv',
]

KNOWN_TEST_CANDIDATES = [
    '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv',
    '/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv',
    '/kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv',
]

available_csvs = sorted(str(p) for p in KAGGLE_INPUT_ROOT.rglob('*.csv')) if KAGGLE_INPUT_ROOT.exists() else []
print('Available CSV files under /kaggle/input:')
for p in available_csvs[:50]:
    print(' -', p)
if len(available_csvs) > 50:
    print(f'... plus {len(available_csvs) - 50} more CSV files')

train_candidates = []
train_candidates.extend(PREFERRED_TRAIN_CANDIDATES)
train_candidates.extend(p for p in available_csvs if Path(p).name == 'train.csv')
# Preserve order while removing duplicates.
train_candidates = list(dict.fromkeys(train_candidates))

TRAIN_PATH = next((p for p in train_candidates if os.path.exists(p)), None)
TEST_PATH = next((p for p in KNOWN_TEST_CANDIDATES + available_csvs if os.path.exists(p) and Path(p).name == 'test.csv'), None)

if TRAIN_PATH is None:
    msg = [
        'Could not find train.csv. This trace generator needs train.csv because it uses the answer column.',
        f'Checked train candidates: {train_candidates}',
        f'Detected test.csv: {TEST_PATH}',
        'If you only attached test.csv, please also add the competition training data/input that contains train.csv.',
        'The path you noted for test.csv is valid for inference/submission notebooks, but not for supervised trace generation.',
    ]
    raise FileNotFoundError('\n'.join(msg))

OUTPUT_DIR = Path('/kaggle/working')
JSONL_PATH = OUTPUT_DIR / 'train_traces_v1.jsonl'
PREVIEW_PATH = OUTPUT_DIR / 'train_traces_v1_preview.csv'
STATS_PATH = OUTPUT_DIR / 'train_traces_v1_stats.csv'
SKIPPED_PATH = OUTPUT_DIR / 'train_traces_v1_skipped.csv'

train = pl.read_csv(TRAIN_PATH)
required_cols = {'id', 'prompt', 'answer'}
missing_cols = required_cols - set(train.columns)
if missing_cols:
    raise ValueError(f'TRAIN_PATH={TRAIN_PATH} is missing required columns: {sorted(missing_cols)}')

print('TRAIN_PATH:', TRAIN_PATH)
print('TEST_PATH:', TEST_PATH)
print('Train shape:', train.shape)
print(train.head())


In [ ]:
def boxed(answer: str) -> str:
    # Do not write f'\boxed{{...}}' directly: '\b' is a Python backspace escape.
    # Building the string from an escaped backslash preserves literal LaTeX \boxed{...}.
    return '\\boxed{' + str(answer).strip() + '}'

def extract_boxed_answer(text: str) -> str | None:
    matches = re.findall(r'\\boxed\{([^}]*)\}', str(text))
    return matches[-1].strip() if matches else None

def classify_problem(prompt: str) -> str:
    p = prompt.lower()
    if 'roman' in p or 'numeral system' in p or 'xliv' in p or 'lxv' in p:
        return 'numeral'
    if 'd = 0.5*g*t^2' in prompt or 'gravitational constant' in p or 'falling distance' in p:
        return 'gravity'
    if 'unit conversion' in p or ('convert the following measurement' in p) or ('convert the following value' in p) or ('becomes' in p and re.search(r'\b(?:cm|mm|m|km|g|kg|mg|l|ml|ft|in|yd|mi)\b', p)):
        return 'unit_conversion'
    return 'other'

def format_answer_like_ground_truth(value: float, answer: str) -> str:
    ans = str(answer).strip()
    if re.fullmatch(r'-?\d+', ans):
        return str(int(round(value)))
    if re.fullmatch(r'-?\d+\.\d+', ans):
        decimals = len(ans.split('.')[-1])
        return f'{value:.{decimals}f}'
    return str(value)

def to_roman(n: int) -> str:
    vals = [(100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'), (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I')]
    out = []
    for v, sym in vals:
        while n >= v:
            out.append(sym)
            n -= v
    return ''.join(out)

# Smoke tests for exact metric-facing string shape.
_smoke = boxed('42')
assert _smoke == r'\boxed{42}', repr(_smoke)
assert extract_boxed_answer(_smoke) == '42'
assert '\x08' not in _smoke, repr(_smoke)


In [ ]:
def build_numeral_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    match = re.search(r'write the number\s+(\d+)\s+in the wonderland numeral system', prompt, flags=re.IGNORECASE)
    if not match:
        return None
    target = int(match.group(1))
    roman = to_roman(target)
    return (
        f'Step 1: The examples match standard Roman numerals.\n'
        f'Step 2: Convert the target number {target} into Roman numerals.\n'
        f'Step 3: Roman numeral form is {roman}.\n'
        f'Final answer: {boxed(answer)}'
    )

def parse_gravity_examples(prompt):
    examples = re.findall(r'For t =\s*([0-9]+(?:\.[0-9]+)?)s,\s*distance =\s*([0-9]+(?:\.[0-9]+)?)', prompt)
    query = re.search(r'determine the falling distance for t =\s*([0-9]+(?:\.[0-9]+)?)s', prompt, flags=re.IGNORECASE)
    if not examples or not query:
        return None
    return [(float(t), float(d)) for t, d in examples], float(query.group(1))

def build_gravity_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    parsed = parse_gravity_examples(prompt)
    if parsed is None:
        return None
    examples, target_t = parsed
    g_estimates = [(2.0 * d) / (t * t) for t, d in examples if t != 0]
    if not g_estimates:
        return None
    g = sum(g_estimates) / len(g_estimates)
    pred = 0.5 * g * target_t * target_t
    formatted_pred = format_answer_like_ground_truth(pred, answer)
    return (
        f'Step 1: Use d = 0.5 * g * t^2.\n'
        f'Step 2: Estimate g from the examples.\n'
        f'Step 3: The inferred g is approximately {g:.4f}.\n'
        f'Step 4: Substitute t = {target_t:.2f}.\n'
        f'Step 5: The computed distance is {formatted_pred}.\n'
        f'Final answer: {boxed(answer)}'
    )

def parse_unit_conversion_examples(prompt):
    examples = re.findall(r'([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)\s+becomes\s+([0-9]+(?:\.[0-9]+)?)', prompt, flags=re.IGNORECASE)
    query = re.search(r'convert the following (?:measurement|value):\s*([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)', prompt, flags=re.IGNORECASE)
    if not examples or not query:
        return None
    return [(float(x), unit, float(y)) for x, unit, y in examples], float(query.group(1)), query.group(2)

def build_unit_conversion_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    parsed = parse_unit_conversion_examples(prompt)
    if parsed is None:
        return None
    examples, target_x, target_unit = parsed
    deltas = [y - x for x, _, y in examples]
    avg_delta = sum(deltas) / len(deltas)
    pred = target_x + avg_delta
    formatted_pred = format_answer_like_ground_truth(pred, answer)
    return (
        f'Step 1: Compare each example input and output in {target_unit}.\n'
        f'Step 2: Infer the hidden conversion rule from the examples.\n'
        f'Step 3: The average shift from the examples is {avg_delta:.4f}.\n'
        f'Step 4: Apply the same rule to {target_x:.2f} {target_unit}.\n'
        f'Step 5: The computed converted value is {formatted_pred}.\n'
        f'Final answer: {boxed(answer)}'
    )


In [ ]:
def build_trace(row):
    category = classify_problem(row['prompt'])
    if category == 'numeral':
        return category, build_numeral_trace(row)
    if category == 'gravity':
        return category, build_gravity_trace(row)
    if category == 'unit_conversion':
        return category, build_unit_conversion_trace(row)
    return category, None

records = []
skipped = []

for row in train.to_dicts():
    category, trace = build_trace(row)
    user_content = row['prompt'] + '\nPlease put your final answer inside \\boxed{}. For example: \\boxed{your answer}'

    if trace is None:
        skipped.append({'id': row['id'], 'category': category, 'answer': row['answer'], 'reason': 'unsupported_or_parse_failed'})
        continue

    records.append({
        'id': row['id'],
        'category': category,
        'answer': row['answer'],
        'messages': [
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': trace},
        ],
        'assistant_text': trace,
        'approx_trace_chars': len(trace),
    })

print('Generated records:', len(records))
print('Skipped records:', len(skipped))


In [ ]:
def validate_records(records):
    if not records:
        raise ValueError('No trace records were generated.')

    failures = []
    for rec in records:
        text = rec['assistant_text']
        expected = str(rec['answer']).strip()
        extracted = extract_boxed_answer(text)
        if '\\boxed{' not in text:
            failures.append((rec['id'], rec['category'], 'missing_literal_boxed', text[-160:]))
        if '\x08' in text:
            failures.append((rec['id'], rec['category'], 'contains_backspace_escape_bug', repr(text[-160:])))
        if extracted != expected:
            failures.append((rec['id'], rec['category'], f'boxed_mismatch expected={expected!r} extracted={extracted!r}', text[-160:]))

    if failures:
        print('Validation failures:', len(failures))
        for failure in failures[:10]:
            print(failure)
        raise AssertionError('Trace validation failed. See failures above.')

    print('Validation passed.')
    print('All assistant traces contain literal \\boxed{...}.')
    print('All boxed answers exactly match train.answer.')
    print('No backspace escape bug detected.')

validate_records(records)


In [ ]:
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

preview_rows = [
    {
        'id': rec['id'],
        'category': rec['category'],
        'answer': rec['answer'],
        'approx_trace_chars': rec['approx_trace_chars'],
        'assistant_preview': rec['assistant_text'][:300],
    }
    for rec in records
]

pd.DataFrame(preview_rows).to_csv(PREVIEW_PATH, index=False)
pd.DataFrame([{'category': k, 'count': v} for k, v in pd.Series([r['category'] for r in records]).value_counts().to_dict().items()]).to_csv(STATS_PATH, index=False)
pd.DataFrame(skipped).to_csv(SKIPPED_PATH, index=False)

print('Saved:', JSONL_PATH)
print('Saved:', PREVIEW_PATH)
print('Saved:', STATS_PATH)
print('Saved:', SKIPPED_PATH)


In [ ]:
print('Category counts:')
if records:
    print(pd.Series([r['category'] for r in records]).value_counts())

print('\nFirst examples by category:')
seen = set()
for rec in records:
    cat = rec['category']
    if cat in seen:
        continue
    seen.add(cat)
    print('\n===', cat, '===')
    print(rec['assistant_text'][:800])

print('\nDone.')
